In [1]:
# Core imports for scraping, timing, and data analysis
from io import StringIO
import time
import os
import requests
import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import pearsonr

---

# Exploratorie

In [3]:
# URL of the page with the list of S&P 500 companies
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Setting the header for the HTTP request
headers = {'User-Agent': 'Mozilla/5.0'}


In [4]:
# Making the HTTP request to get the page content
response = requests.get(url, headers=headers)

# Reading the tables from the HTML page using pandas
tables = pd.read_html(StringIO(response.text))


In [5]:
# Displaying the number of tables found
print(len(tables))   

# Displaying the first rows of the first table
tables[0].head()


2


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [6]:
# Preview the second table found on the page
tables[1].head(3)

,vteS&P 500 companies,vteS&P 500 companies.1
0,Energy,APA Corporation Baker Hughes Chevron Corporati...
1,Materials,Air Products Albemarle Corporation Amcor Avery...
2,Industrials,3M A. O. Smith Allegion Ametek Automatic Data ...


In [10]:
# List the unique company names in the first table
tables[0]['Security'].unique()

<StringArray>
[                    '3M',            'A. O. Smith',    'Abbott Laboratories',
                 'AbbVie',              'Accenture',             'Adobe Inc.',
 'Advanced Micro Devices',        'AES Corporation',                  'Aflac',
   'Agilent Technologies',
 ...
     'Williams Companies',   'Willis Towers Watson',          'Workday, Inc.',
           'Wynn Resorts',            'Xcel Energy',             'Xylem Inc.',
            'Yum! Brands',     'Zebra Technologies',          'Zimmer Biomet',
                 'Zoetis']
Length: 503, dtype: str

In [11]:
# List the unique tickers in the first table
tables[0]['Symbol'].unique()

<StringArray>
[ 'MMM',  'AOS',  'ABT', 'ABBV',  'ACN', 'ADBE',  'AMD',  'AES',  'AFL',
    'A',
 ...
  'WMB',  'WTW', 'WDAY', 'WYNN',  'XEL',  'XYL',  'YUM', 'ZBRA',  'ZBH',
  'ZTS']
Length: 503, dtype: str

In [13]:
# Confirm pd.read_html returned a list of DataFrames
print(type(tables))

<class 'list'>


In [ ]:
# If it's a list, check the type of the first item.
if isinstance(tables, list) and len(tables) > 0:
    print(type(tables[0]))

<class 'pandas.DataFrame'>


In [ ]:
# Transforming tables into pandas DataFrames
df_constituents = tables[0]
df_changes = tables[1]

In [ ]:
# Preview the S&P 500 constituents table
df_constituents.head(3)

In [ ]:
# Preview the S&P 500 changes table
df_changes.head(3)

In [12]:
# Check column types and null counts
df_constituents.info()

<class 'pandas.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Symbol                 503 non-null    str  
 1   Security               503 non-null    str  
 2   GICS Sector            503 non-null    str  
 3   GICS Sub-Industry      503 non-null    str  
 4   Headquarters Location  503 non-null    str  
 5   Date added             503 non-null    str  
 6   CIK                    503 non-null    int64
 7   Founded                503 non-null    str  
dtypes: int64(1), str(7)
memory usage: 31.6 KB


In [13]:
# Converting the 'Date added' column to datetime
df_constituents['Date added'] = pd.to_datetime(df_constituents['Date added'], errors='coerce')


In [14]:
# Adding a new column with the addition year
df_constituents['Year added'] = df_constituents['Date added'].dt.year


---

# Functions

In [62]:
# Function to scrape S&P 500 data
def scrapping_snp500(n, urls):
    # URL of the page with the list of S&P 500 companies
    url = urls

    # Setting the header for the HTTP request
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    # Making the HTTP request to get the page content
    response = requests.get(url, headers=headers)

    # Reading the tables from the HTML page using pandas
    tables = pd.read_html(StringIO(response.text))
    df = tables[n]  # The first table contains the list of S&P 500 companies
    
    # Checking whether the table contains the 'Date added' column
    if n == 0:
        # Converting the 'Date added' column to datetime
        df['Date added'] = pd.to_datetime(df['Date added'], errors='coerce')
        # Adding a new column with the addition year
        df['Year added'] = df['Date added'].dt.year
    else:
        return df
        
    return df


In [63]:
# Function to save a DataFrame to a CSV file
def save_dataframe(df, file_name):
    base_dir = r"C:\Users\leona\github\stock-markets-analytics-zoomcamp-2026\data"
    file_path = os.path.join(base_dir, f"{file_name}.csv")
    df.to_csv(file_path, index=False)



In [ ]:
# Download historical price data, accepting either a ticker list or a DataFrame with a 'Ticker'/'Symbol' column
def get_historical_data(df, start_date, end_date, batch_size=50, sleep_seconds=2):
    """
    Function to get historical data for a list of tickers using yfinance.

    Parameters:
    - df: DataFrame with a 'Ticker' or 'Symbol' column, or a list of tickers
    - start_date: Start date in 'YYYY-MM-DD' format (string)
    - end_date: End date in 'YYYY-MM-DD' format (string)
    - batch_size: number of tickers to download before pausing (default 50)
    - sleep_seconds: pause duration in seconds between batches (default 2)

    Returns:
    - df_result: DataFrame with the historical data
    - failed_tickers: list of tickers that failed to download
    """
    if isinstance(df, pd.DataFrame):
        if 'Ticker' in df.columns:
            tickers_list = df['Ticker'].tolist()
        elif 'Symbol' in df.columns:
            tickers_list = df['Symbol'].tolist()
        else:
            raise KeyError("DataFrame must have a 'Ticker' or 'Symbol' column.")
    elif isinstance(df, list):
        tickers_list = df
    else:
        raise TypeError("df must be a DataFrame or a list of tickers.")

    result = {}
    failed_tickers = []

    for i, ticker in enumerate(tickers_list):
        try:
            data = yf.download(ticker, start=start_date, end=end_date, multi_level_index=False)
            if data.empty:
                failed_tickers.append(ticker)
            else:
                result[ticker] = data
        except Exception:
            failed_tickers.append(ticker)

        if (i + 1) % batch_size == 0:
            time.sleep(sleep_seconds)

    df_result = pd.concat(result, names=['Ticker', 'Date']).reset_index()
    return df_result, failed_tickers

In [66]:
def get_eu_historical_data(df, start_date, end_date, batch_size=50, sleep_seconds=2):
    """
    Function to get historical data for European tickers using yfinance.

    Handles the EU-specific issue where a ticker is missing its exchange
    suffix (e.g. 'ZURN' instead of 'ZURN.SW'): when a download fails or
    comes back empty AND the ticker has no known suffix, it looks up the
    ticker's Country and retries once with the corrected ticker.

    Parameters:
    - df: DataFrame with 'Ticker' and 'Country' columns
    - start_date, end_date: 'YYYY-MM-DD' strings
    - batch_size, sleep_seconds: pacing controls

    Returns:
    - df_result: DataFrame with the historical data
    - report: dict with four lists -
        'fixed'                     -> [(original, corrected), ...] suffix issues auto-fixed
        'missing_suffix_unresolved' -> no suffix AND unknown country (couldn't fix)
        'empty'                     -> already had a suffix, but returned no data
        'error'                     -> [(ticker, error_message), ...] already had a suffix, but raised an exception
    """
    country_suffix_map = {
        'Germany': '.DE', 'France': '.PA', 'Netherlands': '.AS', 'Switzerland': '.SW',
        'United Kingdom': '.L', 'Italy': '.MI', 'Spain': '.MC', 'Sweden': '.ST',
        'Norway': '.OL', 'Denmark': '.CO', 'Finland': '.HE', 'Belgium': '.BR',
        'Austria': '.VI', 'Portugal': '.LS', 'Ireland': '.IR',
    }

    tickers_list = df['Ticker'].tolist()
    country_lookup = df.set_index('Ticker')['Country']

    result = {}
    report = {'fixed': [], 'missing_suffix_unresolved': [], 'empty': [], 'error': []}

    for i, ticker in enumerate(tickers_list):
        # --- attempt 1: ticker as-is ---
        try:
            data = yf.download(ticker, start=start_date, end=end_date, multi_level_index=False)
            error = None if not data.empty else 'empty'
        except Exception as e:
            data, error = None, str(e)

        if error is None:
            result[ticker] = data

        else:
            has_suffix = any(ticker.endswith(suf) for suf in country_suffix_map.values())

            if not has_suffix:
                # --- likely a missing-suffix problem: build the corrected ticker and retry ---
                country = country_lookup.get(ticker)
                suffix = country_suffix_map.get(country) if country is not None else None

                if suffix is None:
                    report['missing_suffix_unresolved'].append(ticker)
                else:
                    fixed_ticker = ticker.replace('.', '-') + suffix  # 'BT.A' -> 'BT-A.L'

                    try:
                        data2 = yf.download(fixed_ticker, start=start_date, end=end_date, multi_level_index=False)
                        if not data2.empty:
                            result[fixed_ticker] = data2
                            report['fixed'].append((ticker, fixed_ticker))
                        else:
                            report['empty'].append(fixed_ticker)
                    except Exception as e2:
                        report['error'].append((fixed_ticker, str(e2)))

            else:
                # already had a suffix -> the failure is NOT about the suffix
                if error == 'empty':
                    report['empty'].append(ticker)
                else:
                    report['error'].append((ticker, error))

        if (i + 1) % batch_size == 0:
            time.sleep(sleep_seconds)

    df_result = pd.concat(result, names=['Ticker', 'Date']).reset_index()
    return df_result, report

In [67]:
# Function to calculate the year-to-date (YTD) growth for each ticker
def get_prices(df, date):
    
    df = df[df['Date'] > date]
    # Pivot the DataFrame to have dates as index and tickers as columns
    close_prices = df.pivot(index= 'Date', columns='Ticker', values='Close')
    # Calculating the first and last prices for each ticker
    first_price = close_prices.bfill().iloc[0]
    last_price = close_prices.ffill().iloc[-1]
    # Calculating the year-to-date growth for each ticker
    growing = (last_price / first_price - 1) * 100
    return growing


In [68]:
# Function to compare the growth of each ticker against the S&P 500
def compare_growth(df):
    index_max_growth = df > df.loc['^GSPC']
    return index_max_growth

In [69]:
# Function to get the names of all tables in a given URL
def scrapping(urls, table):
    # URL of the page with the list of companies
    url = urls

    # Setting the header for the HTTP request
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    # Making the HTTP request to get the page content
    response = requests.get(url, headers=headers)
    
    # Checking whether the table is present in the page content
    check_response = table in response.text
    
    # Reading the tables from the HTML page using pandas
    if check_response == True:
        print(f'Table {table} status: {check_response}')
        # Reading the tables from the HTML page using pandas
        tables = pd.read_html(StringIO(response.text), match=table)
        df = tables[0]  
    
    return df


In [70]:
# Function to get the names of the tables on an HTML page
def get_html_tables_name(urls):
    """
    Function to get the names of the tables on an HTML page.
    
    Parameters:
    - response: HTTP response object containing the page content.
    
    Returns:
    - List with the names of the tables found on the page.
    """
    
    # URL of the page with the list of companies
    url = urls

    # Setting the header for the HTTP request
    headers = {'User-Agent': 'Mozilla/5.0'}

    # Making the HTTP request to get the page content
    response = requests.get(url, headers=headers)
    
    # Reading the tables from the HTML page using pandas
    tables = pd.read_html(StringIO(response.text))
    print(len(tables))

    # Displaying the number of tables found and the first 5 columns of each table
    for i, t in enumerate(tables):
        print(i, t.shape, list(t.columns)[:5])


In [ ]:
def to_yahoo_ticker(row):
    t = str(row['Ticker']).strip()
    suffix = exchange_to_suffix.get(row['Exchange'])
    if suffix is None:
        return None  # exchange not mapped -> stays None for you to check, no guessing
    if t.endswith('.'):        # e.g. "RR.", "BP.", "BA." -> drop the trailing dot
        t = t[:-1]
    t = t.replace(' ', '-').replace('.', '-')  # "NOVO B" / "BT.A" -> share-class notation -> hyphen
    return t + suffix

---

# Questions

### Question 1

In [73]:
# Scrape the S&P 500 constituents table and preview it
df_constituents = scrapping_snp500(0,'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')
df_constituents.head(3)

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded,Year added
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902,1957
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916,2017
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888,1957


In [74]:
# Counting the number of stocks added per year
df_added_per_year = df_constituents.groupby('Year added').size().reset_index(name='Count')
df_added_per_year.head(3)


,Year added,Count
0,1957,52
1,1964,1
2,1965,2


In [75]:
# Finding the year with the highest number of stocks added
df_2020_plus = df_added_per_year[df_added_per_year['Year added'] >= 2020]
df_2020_plus.head(3)


,Year added,Count
51,2020,10
52,2021,10
53,2022,15


In [76]:
# Finding the year with the highest number of stocks added since 2020
year_highest_stokes = df_2020_plus.loc[df_2020_plus['Count'].idxmax()]
year_highest_stokes


Year added    2025
Count           18
Name: 56, dtype: int64

In [77]:
# Counting the number of stocks that have been in the list for more than 20 years
df_over_20_years = df_constituents[df_constituents['Date added'] <= '2006-01-01'].count()
df_over_20_years


Symbol                   218
Security                 218
GICS Sector              218
GICS Sub-Industry        218
Headquarters Location    218
Date added               218
CIK                      218
Founded                  218
Year added               218
dtype: int64

---

### Question 2

In [79]:
# 10 Index Tickers
index_10 = ['^GSPC', '000001.SS', '^HSI',
                '^AXJO', '^NSEI', '^GSPTSE',
                '^GDAXI', '^FTSE', '^N225',
                '^MXX', '^BVSP']

In [ ]:
# Getting S&P 500 historical data from 1950 to 2026
df_index_10, failed_10 = get_historical_data(index_10, '1950-01-01', '2026-08-21')


In [38]:
# Calculating growth for different periods
df_prices_2026 = get_prices(df_index_10, '2026-01-01')
df_prices_2023 = get_prices(df_index_10, '2023-01-01')
df_prices_2021 = get_prices(df_index_10, '2021-01-01')
df_prices_2016 = get_prices(df_index_10, '2016-01-01')



In [39]:
# Comparing tickers' growth relative to the S&P 500
compare_ytd = compare_growth(df_prices_2026)
compare_ytd


Ticker
000001.SS    False
^AXJO        False
^BVSP        False
^FTSE        False
^GDAXI       False
^GSPC        False
^GSPTSE       True
^HSI         False
^MXX         False
^N225         True
^NSEI        False
dtype: bool

In [40]:
# Comparing tickers' growth relative to the S&P 500
compare_10_years = compare_growth(df_prices_2016)
compare_10_years


Ticker
000001.SS    False
^AXJO        False
^BVSP         True
^FTSE        False
^GDAXI       False
^GSPC        False
^GSPTSE      False
^HSI         False
^MXX         False
^N225        False
^NSEI        False
dtype: bool

In [41]:
# Comparing tickers' growth relative to the S&P 500
compare_5_years = compare_growth(df_prices_2021)
compare_5_years


Ticker
000001.SS    False
^AXJO        False
^BVSP        False
^FTSE        False
^GDAXI       False
^GSPC        False
^GSPTSE       True
^HSI         False
^MXX         False
^N225         True
^NSEI        False
dtype: bool

In [42]:
# Comparing tickers' growth relative to the S&P 500
compare_3_years = compare_growth(df_prices_2023)
compare_3_years


Ticker
000001.SS    False
^AXJO        False
^BVSP        False
^FTSE        False
^GDAXI       False
^GSPC        False
^GSPTSE      False
^HSI         False
^MXX         False
^N225         True
^NSEI        False
dtype: bool

In [43]:
# Summarizing the results
print("YTD:", compare_ytd.sum())
print("10 anos:", compare_10_years.sum())
print("5 anos:", compare_5_years.sum())
print("3 anos:", compare_3_years.sum())

YTD: 2
10 anos: 1
5 anos: 2
3 anos: 1


---

### Question 3

In [ ]:
# Download historical price data for the S&P 500 constituents
df_historical_sp500, failed_sp500_constituents = get_historical_data(df_constituents, '1950-01-03', '2016-08-26')

In [90]:
# Getting S&P 500 historical data
df_historical_sp500 = df_index_10[df_index_10['Ticker'] == '^GSPC']
df_historical_sp500 = df_historical_sp500[['Date', 'Close']].reset_index(drop=True)
df_historical_sp500


,Date,Close
0,1950-01-03,16.660000
1,1950-01-04,16.850000
2,1950-01-05,16.930000
3,1950-01-06,16.980000
4,1950-01-09,17.080000
...,...,...
19275,2026-08-14,7785.759766
19276,2026-08-17,7745.060059
19277,2026-08-18,7691.759766
19278,2026-08-19,7707.979980


In [91]:
# Calculating the S&P 500's cumulative maximum value
df_historical_sp500['cummax'] = df_historical_sp500['Close'].cummax()


In [92]:
# Flagging whether the closing price is a new all-time high
df_historical_sp500['is_ath'] = df_historical_sp500['Close'] == df_historical_sp500['cummax']


In [93]:
# Getting the days when the S&P 500 hit a new all-time high
historical_max_days = df_historical_sp500[df_historical_sp500['is_ath']]
historical_max_days[['Date', 'Close']]


,Date,Close
0,1950-01-03,16.660000
1,1950-01-04,16.850000
2,1950-01-05,16.930000
3,1950-01-06,16.980000
4,1950-01-09,17.080000
...,...,...
19223,2026-06-01,7599.959961
19224,2026-06-02,7609.779785
19267,2026-08-04,7736.520020
19270,2026-08-07,7757.640137


In [94]:
historical_max_days = historical_max_days.sort_values('Date').reset_index(drop=True)

results = []

for i in range(len(historical_max_days) - 1):
    peak_date = historical_max_days.loc[i, 'Date']
    peak_price = historical_max_days.loc[i, 'Close']
    next_peak_date = historical_max_days.loc[i + 1, 'Date']

    # get only the days BETWEEN the two peaks (excluding the peak days themselves)
    period = df_historical_sp500[
        (df_historical_sp500['Date'] > peak_date) &
        (df_historical_sp500['Date'] < next_peak_date)
    ]

    if period.empty:
        continue  # the two peaks are consecutive days, no drop in between

    min_price = period['Close'].min()
    min_date = period.loc[period['Close'].idxmin(), 'Date']

    results.append({
        'peak_date': peak_date,
        'peak_price': peak_price,
        'min_date': min_date,
        'min_price': min_price,
        'next_peak_date': next_peak_date
    })

df_corrections = pd.DataFrame(results)
df_corrections.head()

,peak_date,peak_price,min_date,min_price,next_peak_date
0,1950-01-09,17.08,1950-01-10,17.030001,1950-01-11
1,1950-01-11,17.09,1950-01-13,16.670000,1950-02-02
2,1950-02-06,17.32,1950-02-16,16.990000,1950-03-06
3,1950-03-06,17.32,1950-03-09,17.070000,1950-03-15
4,1950-03-16,17.49,1950-03-20,17.440001,1950-03-22


In [95]:
# Calculate the drawdown percentage
df_corrections['drawdown_pct'] = (df_corrections['peak_price'] - df_corrections['min_price']) / df_corrections['peak_price'] * 100

In [96]:
# Filter the DataFrame to include only rows where the drawdown percentage is greater than or equal to 5%
df_significant = df_corrections[df_corrections['drawdown_pct'] >= 5]

In [103]:
# Median drawdown of significant corrections (Question 3 answer)
df_significant['drawdown_pct'].median()

np.float64(7.986357561745869)

In [97]:
# Calculate the duration of each drawdown in days
df_significant['duration_days'] = (df_significant['min_date'] - df_significant['peak_date']).dt.days

In [98]:
# Calculate percentiles for duration and drawdown
duration_percentiles = df_significant['duration_days'].quantile([0.25, 0.5, 0.75])
drawdown_percentiles = df_significant['drawdown_pct'].quantile([0.25, 0.5, 0.75])

print(duration_percentiles)
print(drawdown_percentiles)

0.25    22.00
0.50    40.50
0.75    86.25
Name: duration_days, dtype: float64
0.25     6.234677
0.50     7.986358
0.75    14.019826
Name: drawdown_pct, dtype: float64


---

### Question 4

In [ ]:
# Download AMZN's full historical price data
df_amzn, failed_amzn = get_historical_data(['AMZN'], '2020-10-27', '2026-10-29')

In [108]:
# Preview the downloaded data
df_amzn

,Ticker,Date,Close,High,Low,Open,Volume
0,AMZN,2020-10-27,164.316498,164.582993,160.565002,161.246994,85820000
1,AMZN,2020-10-28,158.139008,163.201004,158.123505,162.464996,111766000
2,AMZN,2020-10-29,160.550507,162.862503,158.199997,160.063507,131930000
3,AMZN,2020-10-30,151.807495,158.350006,150.949997,157.887497,167728000
4,AMZN,2020-11-02,150.223999,153.985504,147.505997,153.087006,145148000
...,...,...,...,...,...,...,...
1468,AMZN,2026-09-02,254.979996,256.239990,253.399994,254.259995,24013300
1469,AMZN,2026-09-03,258.899994,259.500000,256.000000,256.200012,26891500
1470,AMZN,2026-09-04,258.510010,261.119995,255.289993,259.070007,30749900
1471,AMZN,2026-09-08,256.970001,257.989990,254.750000,256.679993,29349000


In [109]:
# Calculating the previous day's closing price for each row of the DataFrame
df_amzn['Close_Day1'] = df_amzn['Close'].shift(1)


In [110]:
# Calculating the next day's closing price for each row of the DataFrame
df_amzn['Close_Day3'] = df_amzn['Close'].shift(-1)


In [111]:
# Calculating the two-day return for each row of the DataFrame
df_amzn['return_2d'] = df_amzn['Close_Day3'] / df_amzn['Close_Day1'] - 1


In [112]:
# Preview the DataFrame with the new shifted-price and return columns
df_amzn

,Ticker,Date,Close,High,Low,Open,Volume,Close_Day1,Close_Day3,return_2d
0,AMZN,2020-10-27,164.316498,164.582993,160.565002,161.246994,85820000,NaN,158.139008,NaN
1,AMZN,2020-10-28,158.139008,163.201004,158.123505,162.464996,111766000,164.316498,160.550507,-0.022919
2,AMZN,2020-10-29,160.550507,162.862503,158.199997,160.063507,131930000,158.139008,151.807495,-0.040038
3,AMZN,2020-10-30,151.807495,158.350006,150.949997,157.887497,167728000,160.550507,150.223999,-0.064319
4,AMZN,2020-11-02,150.223999,153.985504,147.505997,153.087006,145148000,151.807495,152.420502,0.004038
...,...,...,...,...,...,...,...,...,...,...
1468,AMZN,2026-09-02,254.979996,256.239990,253.399994,254.259995,24013300,254.919998,258.899994,0.015613
1469,AMZN,2026-09-03,258.899994,259.500000,256.000000,256.200012,26891500,254.979996,258.510010,0.013844
1470,AMZN,2026-09-04,258.510010,261.119995,255.289993,259.070007,30749900,258.899994,256.970001,-0.007455
1471,AMZN,2026-09-08,256.970001,257.989990,254.750000,256.679993,29349000,258.510010,251.799896,-0.025957


In [113]:
# Check the new columns' types and null counts
df_amzn.info()

<class 'pandas.DataFrame'>
RangeIndex: 1473 entries, 0 to 1472
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype        
---  ------      --------------  -----        
 0   Ticker      1473 non-null   str          
 1   Date        1473 non-null   datetime64[s]
 2   Close       1473 non-null   float64      
 3   High        1473 non-null   float64      
 4   Low         1473 non-null   float64      
 5   Open        1473 non-null   float64      
 6   Volume      1473 non-null   int64        
 7   Close_Day1  1472 non-null   float64      
 8   Close_Day3  1472 non-null   float64      
 9   return_2d   1471 non-null   float64      
dtypes: datetime64[s](1), float64(7), int64(1), str(1)
memory usage: 115.2 KB


In [114]:
# Getting Amazon (AMZN) earnings release dates
ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)
df_earnings = ticker_obj.get_earnings_dates()


In [115]:
# Preview the earnings dates DataFrame
df_earnings

,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2026-10-29 16:00:00-04:00,1.95,NaN,NaN
2026-07-30 16:00:00-04:00,1.83,5.75,215.02
2026-04-29 16:00:00-04:00,1.64,2.78,69.02
2026-02-05 16:00:00-05:00,1.95,1.95,0.22
2025-10-30 16:00:00-04:00,1.56,1.95,25.20
2025-07-31 16:00:00-04:00,1.32,1.68,27.19
2025-05-01 16:00:00-04:00,1.36,1.59,16.77
2025-02-06 16:00:00-05:00,1.48,1.86,25.29
2024-10-31 16:00:00-04:00,1.14,1.43,25.22


In [116]:
# Check column types and null counts
df_earnings.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 25 entries, 2026-10-29 16:00:00-04:00 to 2020-10-29 16:00:00-04:00
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   EPS Estimate  25 non-null     float64
 1   Reported EPS  24 non-null     float64
 2   Surprise(%)   24 non-null     float64
dtypes: float64(3)
memory usage: 800.0 bytes


In [117]:
# Resetting the index of the earnings DataFrame to make 'Date' a column instead of an index
df_earnings = df_earnings.reset_index()

In [118]:
df_earnings['Earnings Date'] = (
    df_earnings['Earnings Date']
    .dt.tz_localize(None)   # remove the timezone
    .dt.normalize()         # zero out the time, keep only the date (00:00:00)
)


In [119]:
# Merging the earnings DataFrame with the AMZN historical prices DataFrame on the 'Date' column
df_merged = df_amzn.merge(
    df_earnings,
    left_on='Date',
    right_on='Earnings Date',
    how='inner'
)

In [120]:
print(len(df_merged))  # expected: 24 (the 25 minus the 2026-10-29 one, which hasn't happened yet)


24


In [121]:
# Filtering the merged DataFrame to include only rows where the 'Surprise(%)' column is greater than 0
df_positives = df_merged[df_merged['Surprise(%)'] > 0]


In [122]:
# Calculating the median of the 'retorno_2d' column for rows where 'Surprise(%)' is greater than 0
return_median = df_positives['return_2d'].median()

In [105]:
# Display the median 2-day return following positive earnings surprises (Question 4 answer)
print(f'Median Return: {return_median}')

NameError: name 'return_median' is not defined

In [124]:
# Calculating the correlation between the 2-day return and the Surprise(%)
correlation = df_merged['return_2d'].corr(df_merged['Surprise(%)'])
print(f'Correlation between 2-day return and Surprise(%): {correlation}')

Correlation between 2-day return and Surprise(%): 0.21906338566924644


In [125]:
# Correlation matrix between 2-day return and earnings surprise magnitude
df_merged[['return_2d', 'Surprise(%)']].corr()

,return_2d,Surprise(%)
return_2d,1.000000,0.219063
Surprise(%),0.219063,1.000000


In [ ]:
# 
r, p_value = pearsonr(df_merged['return_2d'], df_merged['Surprise(%)'])
print(f'r = {r:.3f}, p-value = {p_value:.3f}')

r = 0.219, p-value = 0.304


---

### Question 5

##### Euro Stoxx 50

In [97]:
# Inspect the tables available on the Euro Stoxx 50 Wikipedia page
get_html_tables_name('https://en.wikipedia.org/wiki/EURO_STOXX_50')

16
0 (8, 2) [0, 1]
1 (7, 2) ['Euro Stoxx 50 (DSX)', 'Unnamed: 1']
2 (40, 4) ['Year', 'Year end value', 'Change, in points', 'Change, in\xa0%']
3 (50, 7) ['Ticker', 'Main listing', 'Name', 'Corporate form', 'Registered office']
4 (8, 3) ['Rank', 'Country', 'Stocks']
5 (2, 2) ['vte EURO STOXX 50 companies of the Euro Area', 'vte EURO STOXX 50 companies of the Euro Area.1']
6 (34, 2) ['vteStock market indices', 'vteStock market indices.1']
7 (2, 2) [0, 1]
8 (4, 2) [0, 1]
9 (5, 2) [0, 1]
10 (5, 2) [0, 1]
11 (2, 2) [0, 1]
12 (3, 2) [0, 1]
13 (5, 2) [0, 1]
14 (2, 2) [0, 1]
15 (1, 2) [0, 1]


In [53]:
# Scrape the Euro Stoxx 50 constituents table and preview it
df_scrapping_euro_stoxx_50 = scrapping('https://en.wikipedia.org/wiki/EURO_STOXX_50', 'Ticker')
df_scrapping_euro_stoxx_50.head(3)

Table Ticker status: True


,Ticker,Main listing,Name,Corporate form,Registered office,Sector,Founded
0,ADS.DE,FWB: ADS,Adidas,Aktiengesellschaft,Germany,Consumer Discretionary,1924
1,ADYEN.AS,Euronext Amsterdam: ADYEN,Adyen,Naamloze vennootschap,Netherlands,Financials,2006
2,AD.AS,Euronext Amsterdam: AD,Ahold Delhaize,Naamloze vennootschap,Netherlands,Consumer Staples,2016 (1867)


In [55]:
# Save the Euro Stoxx 50 tickers to CSV
save_dataframe(df_scrapping_euro_stoxx_50, 'stoxx_50_tickers')

In [54]:
# Download historical price data for the Euro Stoxx 50 constituents
df_euro_stoxx_50, df_empty_tickers =get_historical_data(df_scrapping_euro_stoxx_50, '2018-01-01', '2026-10-29')

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

In [102]:
# Check which tickers failed to download
df_empty_tickers

[]

In [103]:
# Preview the downloaded historical data
df_euro_stoxx_50

,Ticker,Date,Close,High,Low,Open,Volume
0,ADS.DE,2018-01-02,152.574478,153.076521,150.840164,152.574478,587067
1,ADS.DE,2018-01-03,153.396027,153.396027,151.570427,152.985269,662050
2,ADS.DE,2018-01-04,155.404175,156.727732,153.806775,154.400089,806719
3,ADS.DE,2018-01-05,157.047226,157.594898,156.088783,156.727740,693116
4,ADS.DE,2018-01-08,157.686157,158.964071,157.366672,158.918442,805948
...,...,...,...,...,...,...,...
109545,WKL.AS,2026-09-03,70.580002,72.040001,68.019997,68.019997,1315841
109546,WKL.AS,2026-09-04,68.860001,71.300003,68.699997,71.260002,1169184
109547,WKL.AS,2026-09-08,67.260002,68.760002,66.800003,68.419998,892551
109548,WKL.AS,2026-09-09,66.800003,67.500000,66.260002,66.800003,1060869


In [91]:
# Save the Euro Stoxx 50 historical data to CSV
save_dataframe(df_euro_stoxx_50, 'euro_stoxx_50_historical_data')

#### STOXX Europe 600

In [122]:
# Inspect the tables available on the STOXX Europe 600 Wikipedia page
get_html_tables_name('https://en.wikipedia.org/wiki/STOXX_Europe_600')

15
0 (10, 2) [0, 1]
1 (26, 4) ['Year', 'No. of points', 'Change (points)', 'Change (%)']
2 (1, 2) [0, 1]
3 (467, 5) ['Ticker', 'Company', 'ICB Sector', 'Country', 'Headquarters']
4 (4, 4) ['Date', 'Added', 'Removed', 'Reason']
5 (7, 3) ['Sector', 'Weight (approx.)', 'Key Components']
6 (34, 2) ['vteStock market indices', 'vteStock market indices.1']
7 (2, 2) [0, 1]
8 (4, 2) [0, 1]
9 (5, 2) [0, 1]
10 (5, 2) [0, 1]
11 (2, 2) [0, 1]
12 (3, 2) [0, 1]
13 (5, 2) [0, 1]
14 (2, 2) [0, 1]


In [18]:
# Scrape the STOXX Europe 600 constituents table and preview it
df_scrapping_stoxx_europe_600 = scrapping('https://en.wikipedia.org/wiki/STOXX_Europe_600', 'Ticker')
df_scrapping_stoxx_europe_600.head(3)

Table Ticker status: True


,Ticker,Company,ICB Sector,Country,Headquarters
0,ZURN,Zurich Insurance Group,Insurance,Switzerland,Zurich
1,GSK,GSK plc,Health Care,United Kingdom,London
2,RIO,Rio Tinto Group,Basic Resources,United Kingdom,London


In [19]:
# Count how many tickers were scraped
df_scrapping_stoxx_europe_600['Ticker'].count()

np.int64(467)

In [20]:
# Count how many distinct countries are represented
df_scrapping_stoxx_europe_600['Country'].nunique()

20

In [ ]:
# Map each exchange name to its Yahoo Finance ticker suffix
exchange_to_suffix = {
    'London Stock Exchange': '.L',
    'Xetra': '.DE',
    'Deutsche Boerse Xetra': '.DE',
    'Nyse Euronext - Euronext Paris': '.PA',
    'SIX Swiss Exchange': '.SW',
    'Nasdaq Omx Nordic': '.ST',
    'Borsa Italiana': '.MI',
    'Euronext Amsterdam': '.AS',
    'Bolsa De Madrid': '.MC',
    'Oslo Bors Asa': '.OL',
    'Omx Nordic Exchange Copenhagen A/S': '.CO',
    'Nasdaq Omx Helsinki Ltd.': '.HE',
    'Warsaw Stock Exchange/Equities/Main Market': '.WA',
    'Nyse Euronext - Euronext Brussels': '.BR',
    'Wiener Boerse Ag': '.VI',
    'Irish Stock Exchange - All Market': '.IR',
    'Nyse Euronext - Euronext Lisbon': '.LS',
}

In [ ]:
# the iShares file comes with 2 header rows before the actual table
df = pd.read_csv(r'C:\Users\leona\github\stock-markets-analytics-zoomcamp-2026\data\E600_holdings.csv', skiprows=2)

In [ ]:
# drop cash/future/collateral-fund rows -> keep only real equities
eq = df[df['Type'] == 'EQUITY'].copy()

In [ ]:
# Build the Yahoo-compatible ticker for every row
eq['Yahoo_Ticker'] = eq.apply(to_yahoo_ticker, axis=1)
eq['Ticker'] = eq.apply(to_yahoo_ticker, axis=1)

600 ações, 0 sem sufixo resolvido


,Yahoo_Ticker,Ticker,Name,Sector,Location,Weight (%)
0,ASML.AS,ASML.AS,ASML HOLDING,Information Technology,Netherlands,4.36
1,HSBA.L,HSBA.L,HSBC HOLDINGS PLC,Financials,United Kingdom,2.33
2,ROP.SW,ROP.SW,ROCHE PS PAR AG,Health Care,Switzerland,1.95
3,SHELL.AS,SHELL.AS,SHELL PLC,Energy,United Kingdom,1.73
4,NOVN.SW,NOVN.SW,NOVARTIS AG,Health Care,Switzerland,1.69


In [ ]:
# Check how many tickers still have no suffix resolved
print(f'{len(eq)} stocks, {eq["Yahoo_Ticker"].isna().sum()} with unresolved suffix')
eq[['Yahoo_Ticker','Ticker','Name','Sector','Location','Weight (%)']].head()

In [45]:
# Preview the full DataFrame
eq

,Ticker,Name,Type,Sector,Asset Class,Market Value,Weight (%),Notional Value,Shares,Price,Country,Exchange,Market Currency,Yahoo_Ticker
0,ASML.AS,ASML HOLDING,EQUITY,Information Technology,Equity,"339,955.20",4.36,"339,955.20",227.00,"1,497.60",Netherlands,Euronext Amsterdam,EUR,ASML.AS
1,HSBA.L,HSBC HOLDINGS PLC,EQUITY,Financials,Equity,"181,517.07",2.33,"181,517.07","10,062.00",18.04,United Kingdom,London Stock Exchange,GBP,HSBA.L
2,ROP.SW,ROCHE PS PAR AG,EQUITY,Health Care,Equity,"151,880.39",1.95,"151,880.39",411.00,369.54,Switzerland,SIX Swiss Exchange,CHF,ROP.SW
3,SHELL.AS,SHELL PLC,EQUITY,Energy,Equity,"134,610.72",1.73,"134,610.72","3,288.00",40.94,United Kingdom,Euronext Amsterdam,EUR,SHELL.AS
4,NOVN.SW,NOVARTIS AG,EQUITY,Health Care,Equity,"131,845.98",1.69,"131,845.98","1,111.00",118.67,Switzerland,SIX Swiss Exchange,CHF,NOVN.SW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
601,WIHL.ST,WIHLBORGS FASTIGHETER,EQUITY,Real Estate,Equity,"1,109.93",0.01,"1,109.93",161.00,6.89,Sweden,Nasdaq Omx Nordic,SEK,WIHL.ST
602,LIGHT.AS,SIGNIFY,EQUITY,Industrials,Equity,"1,091.27",0.01,"1,091.27",71.00,15.37,Netherlands,Euronext Amsterdam,EUR,LIGHT.AS
603,BUCN.SW,BUCHER INDUSTRIES AG,EQUITY,Industrials,Equity,"1,041.02",0.01,"1,041.02",3.00,347.01,Switzerland,SIX Swiss Exchange,CHF,BUCN.SW
604,SESG.PA,SES SA FDR,EQUITY,Communication,Equity,946.57,0.01,946.57,202.00,4.69,France,Nyse Euronext - Euronext Paris,EUR,SESG.PA


In [ ]:
# Rename 'Location' to 'Country' for consistency with the other ticker files
eq.rename(columns={'Location': 'Country'}, inplace=True)

In [ ]:
# Save the STOXX 600 ticker list to CSV
eq[['Ticker', 'Name', 'Sector', 'Country', 'Exchange', 'Weight (%)','Yahoo_Ticker']].to_csv(
    '../data/stoxx600_tickers.csv', index=False
)

In [40]:
# Check for duplicate Yahoo tickers
eq['Yahoo_Ticker'].duplicated().sum()

np.int64(1)

In [46]:
# Download historical price data for the STOXX Europe 600 constituents
df_sotxx_600, df_empty_tickers = get_eu_historical_data(eq, '2018-01-01', '2026-10-29')

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

In [47]:
# Preview the downloaded historical data
df_sotxx_600

,Ticker,Date,Close,High,Low,Open,Volume
0,ASML.AS,2018-01-02,134.299850,134.759781,130.804372,133.333993,1171821
1,ASML.AS,2018-01-03,136.323547,137.473375,135.081742,135.679647,1531424
2,ASML.AS,2018-01-04,137.841309,138.393217,135.817601,137.105416,1606148
3,ASML.AS,2018-01-05,139.681061,139.681061,137.657353,137.979310,1087996
4,ASML.AS,2018-01-08,141.428772,141.520764,139.727022,140.738876,907322
...,...,...,...,...,...,...,...
1266171,KTY.WA,2026-09-03,1227.000000,1245.000000,1200.000000,1226.000000,17198
1266172,KTY.WA,2026-09-04,1230.000000,1239.000000,1221.000000,1228.000000,9999
1266173,KTY.WA,2026-09-08,1257.000000,1270.000000,1227.000000,1248.000000,10904
1266174,KTY.WA,2026-09-09,1226.000000,1265.000000,1226.000000,1257.000000,15949


In [49]:
# Check which tickers failed to download
df_empty_tickers

{'fixed': [], 'missing_suffix_unresolved': [], 'empty': [], 'error': []}

In [48]:
# Save the STOXX Europe 600 historical data to CSV
save_dataframe(df_sotxx_600, 'euro_stoxx_600_historical_data')